# cancer-recon-apoptosis — Step 2: Cancer-restricted target discovery (Colab)

**Goal:** a ranked shortlist of receptors enriched on CANCER cells vs matched NORMAL cells that participate in cell-cell communication — feeds Step 3 (specificity audit) and Step 4 (reward).

**Engine:** CELLxGENE Census (data) → **LIANA+** (Python; runs the CellChat method + CellPhoneDB/NATMI/… + a consensus). No R/Bioconductor.

**Runtime:** CPU is fine. For big pulls use **Runtime → Change runtime type → High-RAM**. **No GPU needed.**

**Data-first discipline:** run **Cell 3 (explore) FIRST** and read its output — it prints the real `disease` strings, cell counts, and whether malignant cells are annotated. Only then run Cell 4 (fetch). Each cell prints `[CELL N] ▶ … ✓`.

**Honest scope:** this finds cancer-*enriched* receptors (ranked), not cancer-*exclusive* ones (those barely exist). Specificity is enforced later by the ΔΔG-vs-normal-homolog reward.

## 1. Clone / pull repo

In [ ]:
#@title Cell 1 — clone / pull repo (idempotent)
print('[CELL 1] ▶ clone or pull repo')
import os
from pathlib import Path
REPO_URL = 'https://github.com/AnshumanAtrey/cancer-recon-apoptosis.git'
REPO_DIR = Path('/content/cancer-recon-apoptosis')
if REPO_DIR.exists():
    print('  pulling latest'); 
    !cd {REPO_DIR} && git pull --ff-only
else:
    !git clone {REPO_URL} {REPO_DIR}
os.chdir(REPO_DIR)
print('  cwd =', os.getcwd())
assert (REPO_DIR / 'scripts' / '03_census_fetch.py').exists(), 'step-2 script missing'
print('[CELL 1] ✓ done')

## 2. Install CELLxGENE Census + scanpy + LIANA

`cellxgene-census` pulls TileDB-SOMA; `liana` is for Step 2b. First install ~2-3 min.

In [ ]:
#@title Cell 2 — install deps (idempotent)
print('[CELL 2] ▶ pip install cellxgene-census scanpy liana')
import importlib.util
need = [m for m in ('cellxgene_census', 'scanpy', 'liana') if importlib.util.find_spec(m) is None]
if need:
    print('  installing:', need)
    !pip install --quiet cellxgene-census scanpy liana
else:
    print('  already installed')
import cellxgene_census, scanpy
print('  cellxgene_census', cellxgene_census.__version__, '| scanpy', scanpy.__version__)
print('[CELL 2] ✓ done')

## 3. EXPLORE — look at the data first

Prints, per candidate tissue: total cells, the available `disease` values (so we use the exact strings), the top `cell_type` values, and whether `malignant cell` / `epithelial cell` are annotated. **Read this before fetching.** If the disease strings in `TARGETS` (top of `scripts/03_census_fetch.py`) don't match what's printed, fix them.

In [ ]:
#@title Cell 3 — explore Census metadata (cheap, ~1-2 min)
print('[CELL 3] ▶ python -u scripts/03_census_fetch.py explore')
import subprocess, sys
proc = subprocess.Popen([sys.executable, '-u', 'scripts/03_census_fetch.py', 'explore'],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='', flush=True)
rc = proc.wait()
print(f'\n[CELL 3] exit={rc}', '✓' if rc==0 else '✗ see log above')

## 4. FETCH — pull tumour + normal AnnData per target

Capped + subsampled (default 20k cells/condition) and saved to `data/cellxgene/<label>__<tumour|normal>.h5ad`. Idempotent (skips existing). If a target reports **0 cells**, the disease string is wrong — fix `TARGETS` in `scripts/03_census_fetch.py` using Cell 3's output, then re-run.

In [ ]:
#@title Cell 4 — fetch tumour + normal cells (minutes)
print('[CELL 4] ▶ python -u scripts/03_census_fetch.py fetch')
import subprocess, sys
proc = subprocess.Popen([sys.executable, '-u', 'scripts/03_census_fetch.py', 'fetch'],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='', flush=True)
rc = proc.wait()
print(f'\n[CELL 4] exit={rc}', '✓' if rc==0 else '✗ see log above')
!ls -la data/cellxgene 2>/dev/null

## 5. Next: Step 2b — LIANA+ communication inference

`scripts/04_liana_communication.py` (written after we confirm the data shape from Cells 3–4): runs LIANA+ on tumour and normal AnnData, compares ligand-receptor communication, and ranks receptors enriched in cancer. Paste the Cell 3 + Cell 4 output back so the analysis is designed from the real data (disease strings, malignant-cell annotation, cell counts).